# 20 — Generate the core T-Box

Renders the two published COSMoS LinkML models as OWL, **one graph per schema**
(decision D1, settled 2026-09-01 — `docs/decisions.md`). Reads `../downloads/`,
which `10_fetch_cosmos.ipynb` must have populated first.

Outputs at repo root:

- `cosmos_bc_v1.ttl`
- `cosmos_sdtm_v1.ttl`

Requires `linkml` (developed against 1.11.1).

## Repairs applied to the source

One repair, and it is not a modelling choice. `cosmos_bc_model.yaml` declares its
namespace prefix with **no trailing separator**, so every IRI generated from it
concatenates into `…biomedical_concept_v1.0BiomedicalConcept`. The SDTM model
declares its own prefix with the slash. See `docs/known-gaps.md` §1a.

The repair cannot be made from a wrapper schema — LinkML rejects a prefix
redeclaration — and `gen-owl` has no prefix-override option, so a patched copy is
built into `../build/`. **`../downloads/` is never modified**: it holds the
upstream bytes the sidecar checksums describe.

The substitution is asserted to match exactly once. If upstream fixes the defect,
this cell fails loudly rather than silently doing nothing — which is the correct
signal to drop the repair.

In [ ]:
DOWNLOADS = "../downloads"
BUILD     = "../build"
PATCHES   = "../patches"
ROOT      = ".."

BC_MODEL   = "cosmos_bc_model.yaml"
SDTM_MODEL = "cosmos_sdtm_model.yaml"

BC_TTL   = "cosmos_bc_v1.ttl"
SDTM_TTL = "cosmos_sdtm_v1.ttl"

BC_NS   = "https://www.cdisc.org/cosmos/biomedical_concept_v1.0"
SDTM_NS = "https://www.cdisc.org/cosmos/sdtm_v1.0"

# (description, exact text to find, replacement). Each must match exactly once.
REPAIRS = [
    (
        "known-gaps.md 1a: BC namespace prefix has no trailing separator",
        "  cosmos_bc: https://www.cdisc.org/cosmos/biomedical_concept_v1.0\n",
        "  cosmos_bc: https://www.cdisc.org/cosmos/biomedical_concept_v1.0/\n",
    ),
]

In [ ]:
import difflib
from pathlib import Path

Path(BUILD).mkdir(parents=True, exist_ok=True)
Path(PATCHES).mkdir(parents=True, exist_ok=True)

original = Path(DOWNLOADS, BC_MODEL).read_text(encoding="utf-8")
patched = original

for description, find, replace in REPAIRS:
    hits = patched.count(find)
    if hits != 1:
        raise RuntimeError(
            f"repair matched {hits} times, expected exactly 1 — {description}"
        )
    patched = patched.replace(find, replace)
    print(f"applied: {description}")

patched_path = Path(BUILD, "cosmos_bc_model.patched.yaml")
patched_path.write_text(patched, encoding="utf-8")

diff = "".join(
    difflib.unified_diff(
        original.splitlines(keepends=True),
        patched.splitlines(keepends=True),
        fromfile=f"downloads/{BC_MODEL}",
        tofile=f"build/cosmos_bc_model.patched.yaml",
    )
)
Path(PATCHES, "cosmos_bc_prefix.patch").write_text(diff, encoding="utf-8")

print()
print(diff)

## Generate

**Every generator option is passed explicitly, and that is not tidiness.**
`OwlSchemaGenerator`'s Python defaults are not the `gen-owl` CLI defaults: the
class defaults to `metaclasses=True` and `type_objects=True`, which the CLI turns
off. Constructing the generator with its own defaults produces a materially
different graph — measured here, 551 instead of 512 triples for the BC model,
with **every** datatype property emitted as an `owl:ObjectProperty` (13 datatype
properties became 0). LinkML additionally warns that three cardinality-axiom
defaults will change in a future release.

So the option set below is pinned to the CLI behaviour at linkml 1.11.1 and
verified to reproduce it triple-for-triple. A future diff in the output then
means a source change, not a toolchain change.

`ontology_uri_suffix` is left at the CLI default, which makes the ontology IRI
`https://www.cdisc.org/cosmos/biomedical_concept_v1.0.owl.ttl` — a file name
standing in for an ontology IRI. That is what the mechanical rendering produces;
whether to override it is decision D6, open.

In [ ]:
from linkml.generators.owlgen import MetadataProfile, OwlSchemaGenerator
from pathlib import Path

# Pinned to the gen-owl CLI behaviour at linkml 1.11.1. Verified to reproduce the
# CLI output triple-for-triple. Do not rely on the class defaults; they differ.
OWL_OPTIONS = {
    "metadata_profiles": [MetadataProfile.linkml],
    "metadata": True,
    "mergeimports": True,
    "metaclasses": False,
    "type_objects": False,
    "ontology_uri_suffix": ".owl.ttl",
    "add_ols_annotations": True,
    "add_root_classes": False,
    "assert_equivalent_classes": False,
    "enum_iri_separator": "#",
    "enum_inherits_as_subclass_of": False,
    "mixins_as_expressions": False,
    "skip_abstract_class_as_unionof_subclasses": False,
    "use_native_uris": True,
    "useuris": True,
    "xsd_anyuri_as_iri": False,
    "default_permissible_value_type": "http://www.w3.org/2002/07/owl#Class",
    "skip_vacuous_min_zero_cardinality_axioms": False,
    "skip_vacuous_local_range_axioms": False,
    "consolidate_cardinality_axioms": False,
}

SOURCES = {
    BC_TTL:   Path(BUILD, "cosmos_bc_model.patched.yaml"),
    SDTM_TTL: Path(DOWNLOADS, SDTM_MODEL),
}

from rdflib import Graph

generated = {}

for target, source in SOURCES.items():
    generator = OwlSchemaGenerator(str(source), **OWL_OPTIONS)
    graph = Graph().parse(data=generator.serialize(), format="turtle")
    generated[target] = graph
    print(f"{target:20s} {len(graph):>6,} triples generated  <- {source}")

## Author the ontology header — decision D7, option A

LinkML emits an ontology node carrying nothing but `rdf:type owl:Ontology` and
`rdfs:label`. Everything a consumer needs to cite, version or trace the artifact
is authored, exactly as `usdm-rdf` authors its own header.

**D7 settled 2026-09-01, option A.** The *ontology* is named under
`https://w3id.org/cdisc/cosmos/`, one segment per graph. The *terms* keep the
IRIs CDISC published — nothing in either model is renamed. So this repo mints a
name for its own rendering and none for anyone else's concepts, which is what
keeps the core layer mechanical.

CDISC's declared schema id is preserved where it matters: as the namespace every
term IRI still sits in, recorded explicitly as `vann:preferredNamespaceUri`. Note
that the ontology IRI and the term namespace therefore differ — that is the point
of option A, not an oversight.

`owl:versionIRI` is bare-numeric in-namespace (`…/cosmos/bc/0.1.0`), carried over
from `usdm-rdf` decision D3: one generic w3id rewrite rule then dereferences every
release, including past ones, with no per-release PR. Annotation predicates are
declared `owl:AnnotationProperty` (`usdm-rdf` decision D2) so derived
serializations add no declarations the canonical graph lacks.

`dcterms:created` is fixed at first publication and never advances;
`dcterms:modified` tracks the release.

In [ ]:
VERSION = "0.1.0"
CREATED = "2026-09-01"
MODIFIED = "2026-09-01"
CREATOR = "Kerstin Forsberg"
LICENSE = "https://opensource.org/licenses/MIT"

ONTOLOGIES = {
    BC_TTL: {
        "iri": "https://w3id.org/cdisc/cosmos/bc/",
        "title": "CDISC COSMoS Biomedical Concepts (RDF/OWL rendering)",
        "prefix": "cosmos_bc",
        "namespace": BC_NS + "/",
        "sidecar": ".fetch_meta_bc_model.json",
        "description": (
            "A mechanical OWL rendering of the LinkML schema CDISC publishes as "
            "model/cosmos_bc_model.yaml in cdisc-org/COSMoS: the Biomedical "
            "Concept, Data Element Concept and Coding classes, their slots, "
            "patterns and cardinality, and the result-scale and data-type "
            "enumerations. Class and property IRIs are the ones the published "
            "schema declares; this rendering does not rename them."
        ),
    },
    SDTM_TTL: {
        "iri": "https://w3id.org/cdisc/cosmos/sdtm/",
        "title": "CDISC COSMoS SDTM Dataset Specializations (RDF/OWL rendering)",
        "prefix": "cosmos_sdtm",
        "namespace": SDTM_NS + "/",
        "sidecar": ".fetch_meta_sdtm_model.json",
        "description": (
            "A mechanical OWL rendering of the LinkML schema CDISC publishes as "
            "model/cosmos_sdtm_model.yaml in cdisc-org/COSMoS: the SDTM group, "
            "variable, relationship and codelist classes, their required and "
            "ordered slots, identifier patterns, and the controlled enumerations "
            "- including the reification vocabulary (predicate terms, linking "
            "phrases) and the Define-XML origin terminology with its NCIt "
            "meanings. Class and property IRIs are the ones the published schema "
            "declares; this rendering does not rename them."
        ),
    },
}

COMMENT = (
    "Mechanical rendering of a published CDISC COSMoS LinkML schema. "
    "Draft - not a normative CDISC artifact."
)

In [ ]:
import json

from rdflib import Literal, Namespace, URIRef
from rdflib.namespace import DCTERMS, OWL, RDF, RDFS, SKOS, XSD

VANN = Namespace("http://purl.org/vocab/vann/")

ANNOTATION_PROPERTIES = [
    DCTERMS.title,
    DCTERMS.description,
    DCTERMS.creator,
    DCTERMS.created,
    DCTERMS.modified,
    DCTERMS.license,
    DCTERMS.source,
    DCTERMS.bibliographicCitation,
    VANN.preferredNamespacePrefix,
    VANN.preferredNamespaceUri,
]

for target, spec in ONTOLOGIES.items():
    graph = generated[target]

    meta = json.loads(Path(DOWNLOADS, spec["sidecar"]).read_text(encoding="utf-8"))

    existing = list(graph.subjects(RDF.type, OWL.Ontology))
    if len(existing) != 1:
        raise RuntimeError(f"{target}: expected 1 owl:Ontology node, found {len(existing)}")
    graph.remove((existing[0], None, None))

    ontology = URIRef(spec["iri"])
    graph.add((ontology, RDF.type, OWL.Ontology))
    graph.add((ontology, RDFS.label, Literal(spec["title"])))
    graph.add((ontology, RDFS.comment, Literal(COMMENT)))
    graph.add((ontology, DCTERMS.title, Literal(spec["title"])))
    graph.add((ontology, DCTERMS.description, Literal(spec["description"])))
    graph.add((ontology, DCTERMS.creator, Literal(CREATOR)))
    graph.add((ontology, DCTERMS.created, Literal(CREATED, datatype=XSD.date)))
    graph.add((ontology, DCTERMS.modified, Literal(MODIFIED, datatype=XSD.date)))
    graph.add((ontology, DCTERMS.license, URIRef(LICENSE)))
    graph.add((ontology, DCTERMS.source, URIRef(meta["raw_url"])))
    graph.add((
        ontology,
        DCTERMS.bibliographicCitation,
        Literal(f"Forsberg, K. ({CREATED[:4]}). {spec['title']}. {spec['iri']}"),
    ))
    graph.add((ontology, OWL.versionIRI, URIRef(spec["iri"] + VERSION)))
    graph.add((ontology, OWL.versionInfo, Literal(f"v{VERSION}")))
    graph.add((ontology, VANN.preferredNamespacePrefix, Literal(spec["prefix"])))
    graph.add((ontology, VANN.preferredNamespaceUri, URIRef(spec["namespace"])))

    for annotation in ANNOTATION_PROPERTIES:
        graph.add((annotation, RDF.type, OWL.AnnotationProperty))

    graph.bind(spec["prefix"], spec["namespace"])
    graph.bind("dcterms", DCTERMS)
    graph.bind("vann", VANN)
    graph.bind("skos", SKOS)
    graph.bind("NCIT", "http://purl.obolibrary.org/obo/NCIT_")

    graph.serialize(destination=str(Path(ROOT, target)), format="turtle")
    print(f"{target:20s} {len(graph):>6,} triples  ontology {spec['iri']}  versionIRI {spec['iri']}{VERSION}")

## Confirm what was generated

Counts are printed, not asserted — they move when the pinned commit moves.
`30_validate.ipynb` holds the baselines and the conformance checks.

Note that most of the `owl:Class` count is enum permissible values rather than
concepts, so both figures are reported separately.

In [ ]:
from rdflib import Graph
from rdflib.namespace import OWL, RDF

for target in (BC_TTL, SDTM_TTL):
    g = Graph().parse(str(Path(ROOT, target)), format="turtle")
    classes = {str(s) for s in g.subjects(RDF.type, OWL.Class)}
    pvs = {c for c in classes if "#" in c}
    object_props = len(set(g.subjects(RDF.type, OWL.ObjectProperty)))
    data_props = len(set(g.subjects(RDF.type, OWL.DatatypeProperty)))
    print(f"{target}")
    print(f"    triples           {len(g):>6,}")
    print(f"    owl:Class         {len(classes):>6,}   ({len(classes) - len(pvs)} top-level + {len(pvs)} permissible values)")
    print(f"    objectProperty    {object_props:>6,}")
    print(f"    datatypeProperty  {data_props:>6,}")

## Provenance

Both deliverables are generated from the models at the commit pinned in
`10_fetch_cosmos.ipynb`, recorded in `../downloads/.fetch_meta_bc_model.json` and
`../downloads/.fetch_meta_sdtm_model.json`. The single repair applied to the BC
model is written to `../patches/cosmos_bc_prefix.patch` by this notebook, so the
patch cannot drift from what was actually applied.

`../build/` is gitignored — it holds the patched copy, which is derived.

The ontology header is authored, not generated: it describes *this rendering*,
not the standard. `dcterms:source` is read from the fetch sidecar, so it always
names the exact pinned file the graph was built from.